# FloodSense data cleaning

Prepare the 2015–2017 training data for modeling. The held-out 2019 Imelda test set is not loaded or used here.

In [2]:
import sys
from pathlib import Path

project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "config.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the FloodSense project root from the notebook working directory.")
sys.path.insert(0, str(project_root))

from src.config import CLEAN_TRAINING_PATH
from src.data import (
    add_grid_cell,
    clean_training,
    keep_developed_negatives,
    load_training,
    remove_duplicates,
)


## 1. Load the training data

Load only the development training CSV and inspect its row/column shape and missing-value counts before changing it.

In [3]:
training_df = load_training()


Training data shape: (85738, 14)
Missing values by column:
year                   0
storm_name             0
storm_date             0
lat                    0
lon                    0
weight                 0
elevation_m            0
slope_degrees          0
distance_to_bayou_m    0
paved_pct              0
land_cover_type        0
rainfall_inches        0
flooded                0
imperviousness         0


## 2. Remove duplicate location-storm records

A location should appear at most once for a given storm, so remove rows sharing the same latitude, longitude, and storm name.

In [4]:
deduplicated_df = remove_duplicates(training_df)


Duplicate rows removed: 8863


## 3. Keep informative non-flooded locations

Retain all flooded examples, but keep non-flooded examples only in developed land-cover areas, where urban flood-risk comparisons are useful.

In [5]:
developed_df = keep_developed_negatives(deduplicated_df)


Class counts before filtering:
flooded
0    42869
1    34006
Class counts after filtering:
flooded
0    32559
1    34006


## 4. Add geographic grid IDs

Round coordinates to 0.01-degree cells and create a `grid_id`. This identifier supports later area-based splitting and is not a model feature.

In [6]:
gridded_df = add_grid_cell(developed_df, cell_deg=0.01)
gridded_df[["lat", "lon", "grid_id"]].head()


,lat,lon,grid_id
0,29.862192,-95.294884,2986_-9529
1,29.694897,-95.299593,2969_-9530
2,29.684477,-95.518639,2968_-9552
3,29.773975,-95.466208,2977_-9547
4,29.819542,-95.391873,2982_-9539


## 5. Run and save the complete cleaning pipeline

`clean_training` applies deduplication, negative-example filtering, and grid assignment in order, then writes the final training table to the configured processed-data path.

In [7]:
clean_df = clean_training(training_df)
print(f"Cleaned data shape: {clean_df.shape}")
print(f"Saved file: {CLEAN_TRAINING_PATH}")


Duplicate rows removed: 8863
Class counts before filtering:
flooded
0    42869
1    34006
Class counts after filtering:
flooded
0    32559
1    34006
Saved cleaned training data to C:\Users\ssaifudd\Downloads\AI-Flood Project\data\processed\training_clean.csv
Cleaned data shape: (66565, 15)
Saved file: C:\Users\ssaifudd\Downloads\AI-Flood Project\data\processed\training_clean.csv
